In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [ ]:
name='DataExport132110.csv'
path='esight_data/'+name
df = pd.read_csv(path)

In [ ]:
df.head()

In [ ]:
#before columns name reduction: makes use of check_duplicate_prefixes
def get_repeated_columns(df, prefixes):
    # Always include timestamp column
    selected_cols = ['From Timestamp']
    
    # Add columns that start with any of the specified prefixes
    for col in df.columns:
        if any(col.startswith(prefix) for prefix in prefixes):
            selected_cols.append(col)
    
    # Create new dataframe with selected columns
    df_repeats = df[selected_cols].copy()
    
    # Print info about selected columns
    # print(f"Selected {len(selected_cols)-1} columns:")
    # for col in selected_cols:
    #     if col != 'From Timestamp':
    #         print(f"  - {col}")
    
    return df_repeats

# Define the prefixes to look for
prefixes = ['meter1','meter2']

# Create df_repeats
df_repeats = get_repeated_columns(df, prefixes)

# df_repeats.head()

def drop_specific_columns(df, patterns_to_drop=None):
    """
    Drop specific columns from a DataFrame using regex patterns.
    
    This function is designed to remove problematic or unwanted columns from a DataFrame
    using regex pattern matching. It's particularly useful when dealing with complex
    column names that may contain special characters or varying formats.
    
    Args:
        df (pandas.DataFrame): The input DataFrame to process
        patterns_to_drop (list): List of regex patterns for columns to drop. 
                                If None, uses default patterns.
    
    Returns:
        pandas.DataFrame: DataFrame with specified columns removed
        bool: Success status of the operation
    
    Example:
        df, success = drop_specific_columns(df)
        if success:
            print("Columns successfully dropped")
    """
    # Default patterns if none provided
    if patterns_to_drop is None:
        patterns_to_drop = [
        ]
    
    # Find all matching columns
    cols_to_drop = []
    for pattern in patterns_to_drop:
        matching_cols = df.columns[df.columns.str.contains(pattern, regex=True)]
        cols_to_drop.extend(matching_cols)
    
    # Drop the columns
    df = df.drop(columns=cols_to_drop)
    
    # Verify the drop was successful
    remaining_columns = [col for col in df.columns 
                       if any(pattern.split('.*')[0] in col 
                           for pattern in patterns_to_drop)]
    
    return df

# Example usage:
df = drop_specific_columns(df)


# Preprocess


In [ ]:
def clean_column_names(df):
    # Create a dictionary of old_name: new_name for columns except 'timestamp'
    new_names = {}
    for col in df.columns:
        if col != 'timestamp':
            # First split on '-' and take the first part (if there is a dash)
            first_part = col.split('-')[0].strip()
            # Then split that on '_' and take first two parts
            parts = first_part.split('_')
            if len(parts) >= 2:
                new_name = f"{parts[0]}_{parts[1]}"
                new_names[col] = new_name
            else:
                new_names[col] = first_part
    
    # Rename the columns
    df = df.rename(columns=new_names)
    return df

# Test it:
# print("\nBefore:")
# print(df.columns.tolist()[:5])
# df = clean_column_names(df)
# print("\nAfter:")
# print(df.columns.tolist()[:5])


In [ ]:
# Check is any columns have the same first 8 letters.

def check_duplicate_prefixes(df):
    # Get all column names except timestamp
    cols = [col for col in df.columns if col != 'timestamp']
    
    # Create dictionary with first 8 letters as key and full column names as values
    prefix_dict = {}
    for col in cols:
        prefix = col[:8]  # Get first 8 letters
        if prefix in prefix_dict:
            prefix_dict[prefix].append(col)
        else:
            prefix_dict[prefix] = [col]
    
    # Filter for prefixes with more than one column
    duplicates = {k: v for k, v in prefix_dict.items() if len(v) > 1}
    
    if duplicates:
        print("\nFound columns with same first 8 letters:")
        for prefix, columns in duplicates.items():
            print(f"\nPrefix '{prefix}':")
            for col in columns:
                print(f"  - {col}")
        return True
    else:
        print("\nNo columns found with same first 8 letters")
        return False
    
# check_duplicate_prefixes(df)


In [ ]:
# There are some exports form HWM that are old and are still part of the esight export but they have NaN data. so we remove these.
def remove_all_nan_columns(df):
    # Find columns that are all NaN (excluding timestamp column)
    all_nan_cols = df.columns[df.isna().all()].tolist()
    
    # Remove timestamp from the list if it's there
    if 'timestamp' in all_nan_cols:
        all_nan_cols.remove('timestamp')
    
    # Print the columns being removed
    if all_nan_cols:
        print(f"Removing {len(all_nan_cols)} columns that contain all NaN values:")
        print(all_nan_cols)
    
    # Drop these columns and return the cleaned dataframe
    return df.drop(columns=all_nan_cols)

# Modify your load_and_preprocess function:
def load_and_preprocess(file_path):
    # Read CSV file
    df = pd.read_csv(file_path)
    
    # Ensure timestamp is sorted - use inplace=True for rename
    df.rename(columns={"From Timestamp": "timestamp"}, inplace=True)

    # Convert timestamp with specific format (DD/MM/YYYY HH:MM:SS)
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d/%m/%Y %H:%M:%S')
    
    # Clean column names
    df = clean_column_names(df)
    
    # Remove columns with all NaN values
    df = remove_all_nan_columns(df)
    
    # Sort by timestamp
    df = df.sort_values('timestamp')
    
    return df

# Test it
df = load_and_preprocess(path)
print(f"Number of remaining columns: {len(df.columns)}")


In [ ]:
df.head()
# df.info()

In [ ]:
df.to_csv(f'processed_data\{name}')